### 1.读取所有 .jsonl
### 2.每一行转成Document
### 3.创建db
### 4.批量 add_documents入库
### 5.评估

In [3]:
from pathlib import Path
data_dir = Path.cwd()
print(data_dir)
data_dir = data_dir / "generated_question_bank"
print(data_dir)
# for file in data_dir.iterdir():
#     print(file)
for file in data_dir.glob("*technical.jsonl"):
    print(f"jsonl:{file}")

e:\2026\字节和我的心脏只有一个可以跳动\实战八股\rag_real
e:\2026\字节和我的心脏只有一个可以跳动\实战八股\rag_real\generated_question_bank
jsonl:e:\2026\字节和我的心脏只有一个可以跳动\实战八股\rag_real\generated_question_bank\cpp_technical.jsonl
jsonl:e:\2026\字节和我的心脏只有一个可以跳动\实战八股\rag_real\generated_question_bank\cs_fundamentals_technical.jsonl
jsonl:e:\2026\字节和我的心脏只有一个可以跳动\实战八股\rag_real\generated_question_bank\embedded_technical.jsonl
jsonl:e:\2026\字节和我的心脏只有一个可以跳动\实战八股\rag_real\generated_question_bank\frontend_technical.jsonl
jsonl:e:\2026\字节和我的心脏只有一个可以跳动\实战八股\rag_real\generated_question_bank\go_technical.jsonl
jsonl:e:\2026\字节和我的心脏只有一个可以跳动\实战八股\rag_real\generated_question_bank\java_technical.jsonl
jsonl:e:\2026\字节和我的心脏只有一个可以跳动\实战八股\rag_real\generated_question_bank\llm_core_tech_technical.jsonl
jsonl:e:\2026\字节和我的心脏只有一个可以跳动\实战八股\rag_real\generated_question_bank\python_backend_technical.jsonl
jsonl:e:\2026\字节和我的心脏只有一个可以跳动\实战八股\rag_real\generated_question_bank\python_technical.jsonl


In [4]:
from langchain_core.documents import Document

def json_to_document(obj:dict , source_file: str) -> Document:
    """ 
    把一行json转换成Document格式
    """
    role = obj.get("role", "")
    topic = obj.get("topic", "")
    chunk_type = obj.get("chunk_type", "") 
    content = obj.get("question", "")
    key_points = obj.get("reference_points", [])
    #related_topics = obj.get("related_topics", [])
    tags = obj.get("tags", [])

    page_content = (
        f"[知识点]: {topic}\n"
        f"[知识分类]: {role}\n"
        f"[内容]: {content}\n"
        f"[关键点]: {', '.join(key_points)}\n"
        #f"[相关主题]: {', '.join(related_topics)}\n"
        f"[标签]: {', '.join(tags)}\n"
    )

    metadata = {
        "source": source_file,
        "role": role,
        "topic": topic,
        "chunk_type": chunk_type,   
    }

    return Document(page_content=page_content, metadata=metadata)


In [5]:
import json
def load_documents_and_ids_from_jsonl(file_path:Path) -> list[dict]:
    """
    打开文件地址，读取文件内容，并把每一行的 JSON 字符串转换成 Document 对象，最后返回一个 Document 对象的列表。
    """
    documents = []
    ids = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            doc = json_to_document(obj,source_file=file_path.name)
            documents.append(doc)
            doc_id = f"{file_path.stem}_{len(documents)}"
            ids.append(doc_id)
            
    return documents,ids

In [6]:
path = Path(r"E:\2026\字节和我的心脏只有一个可以跳动\实战八股\rag_real\generated_question_bank\cs_fundamentals_technical.jsonl")
documents,ids = load_documents_and_ids_from_jsonl(path)
print(documents[0])

page_content='[知识点]: Arrays
[知识分类]: cs_fundamentals
[内容]: 请解释数组（Array）在内存中的存储方式，并比较其与链表（Linked List）在访问、插入和删除操作上的时间复杂度差异。
[关键点]: 数组在内存中是连续存储的，支持 O(1) 的随机访问。, 链表在内存中是非连续存储，节点通过指针连接，访问需要 O(n) 时间。, 数组插入和删除平均时间复杂度为 O(n)，链表为 O(1)（已知位置时）。, 数组适合频繁访问场景，链表适合频繁插入删除场景。, 内存碎片和缓存局部性：数组更高效。
[标签]: arrays, memory, time-complexity, data-structures
' metadata={'source': 'cs_fundamentals_technical.jsonl', 'role': 'cs_fundamentals', 'topic': 'Arrays', 'chunk_type': ''}


In [7]:
technical_files = list(data_dir.glob("*technical.jsonl"))
all_docs = []
all_ids = []
for file in technical_files:
    docs, ids = load_documents_and_ids_from_jsonl(file)
    all_docs.extend(docs)
    all_ids.extend(ids)

print(all_docs[0])
print(all_ids)
print(f"total docs: {len(all_docs)}")

page_content='[知识点]: C++ Syntax
[知识分类]: cpp
[内容]: 请解释 C++ 中 `const` 关键字在不同上下文中的作用，并给出一个工程实践中常见的使用场景。
[关键点]: 修饰变量：表示常量，不可修改，可用于编译期常量或运行时常量。, 修饰指针/引用：区分顶层 const 和底层 const，影响拷贝语义。, 修饰成员函数：保证函数不修改对象状态，可被 const 对象调用。, 工程场景：常用于接口设计，如 const 成员函数、const 引用参数，提高代码安全性和可读性。
[标签]: const, 语法, 工程实践
' metadata={'source': 'cpp_technical.jsonl', 'role': 'cpp', 'topic': 'C++ Syntax', 'chunk_type': ''}
['cpp_technical_1', 'cpp_technical_2', 'cpp_technical_3', 'cpp_technical_4', 'cpp_technical_5', 'cpp_technical_6', 'cpp_technical_7', 'cpp_technical_8', 'cpp_technical_9', 'cpp_technical_10', 'cpp_technical_11', 'cpp_technical_12', 'cpp_technical_13', 'cpp_technical_14', 'cpp_technical_15', 'cpp_technical_16', 'cpp_technical_17', 'cpp_technical_18', 'cpp_technical_19', 'cpp_technical_20', 'cpp_technical_21', 'cpp_technical_22', 'cpp_technical_23', 'cpp_technical_24', 'cpp_technical_25', 'cpp_technical_26', 'cpp_technical_27', 'cpp_technical_28', 'cpp_technical_29', 'cpp_technical_30', 'cpp_technical_31', 'c

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

emb = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs = {"device":"cuda"},
    encode_kwargs={"normalize_embeddings": True}
)
persist_dir = "Chroma"
collection_name = "questions"
db = Chroma(
    collection_name=collection_name,
    persist_directory=persist_dir,
    embedding_function=emb
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7697.48it/s]


In [8]:
db.add_documents(all_docs, ids=all_ids)
print(db._collection.count())

KeyboardInterrupt: 

In [11]:
batch_size = 1000
for i in range(0,len(all_docs),batch_size):
    batch_docs = all_docs[i:i+batch_size]
    batch_ids = all_ids[i:i+batch_size]

    db.add_documents(batch_docs, ids=batch_ids)
print(db._collection.count())

6976


In [17]:
result = db._collection.get(
    ids = ["cpp_technical_1"]
)
print(type(result))
for doc_id , doc_text , mete in zip(result["ids"] , result["documents"],result["metadatas"]):
    print(f"ID: {doc_id}\npage_content:\n{doc_text}\nmetedata: {mete}\n")

<class 'dict'>
ID: cpp_technical_1
page_content:
[知识点]: C++ Syntax
[知识分类]: cpp
[内容]: 请解释 C++ 中 `const` 关键字在不同上下文中的作用，并给出一个工程实践中常见的使用场景。
[关键点]: 修饰变量：表示常量，不可修改，可用于编译期常量或运行时常量。, 修饰指针/引用：区分顶层 const 和底层 const，影响拷贝语义。, 修饰成员函数：保证函数不修改对象状态，可被 const 对象调用。, 工程场景：常用于接口设计，如 const 成员函数、const 引用参数，提高代码安全性和可读性。
[标签]: const, 语法, 工程实践

metedata: {'source': 'cpp_technical.jsonl', 'chunk_type': '', 'role': 'cpp', 'topic': 'C++ Syntax'}



In [ ]:
import math
def get_relevant_ids(gold:list[dict], relevant_threshold: int) -> set:
    revelant_ids = set()
    for doc_id , rel in gold.items():
        if rel >= relevant_threshold:
            revelant_ids.add(doc_id)
    return revelant_ids

def precision_at_k(retrieved_ids,gold:list[dict],k=10,relevant_threshold=2):
    """
    Precision@k = top-k 里命中 gold 的数量 / k。
    除以的是检索topk的数量
    """
    topk = retrieved_ids[:k]
    relevant_gold_ids = get_relevant_ids(gold,relevant_threshold)
    hit_count = sum(1 for doc_id in topk if doc_id in relevant_gold_ids)
    return hit_count / k

def recall_at_k(retrieved_ids,gold:list[dict],k=10,relevant_threshold=2):
    """
    Recall@k = top-k 里命中的 gold 数量 / 所有应命中的 gold 数量。
    除以的是相关chunk的数量
    """
    topk = retrieved_ids[:k]
    relevant_gold_ids = get_relevant_ids(gold,relevant_threshold)
    if len(get_relevant_ids) == 0:
        return 0
    hit_count = sum(1 for doc_id in topk if doc_id in relevant_gold_ids) 
    return hit_count / len(relevant_gold_ids)


def mrr_at_k(retrieved_ids, gold, k=5, relevant_threshold=2):
    """MRR@k = 第一个命中结果排名的倒数。第一名命中就是 1，第二名命中就是 1/2。"""
    relevant_ids = get_relevant_ids(gold, relevant_threshold)
    for rank, doc_id in enumerate(retrieved_ids[:k], start=1):
        if doc_id in relevant_ids:
            return 1 / rank
    return 0


def dcg_at_k(retrieved_ids, gold:dict, k=5):
    """DCG@k = 按排名折损后的相关性得分。越相关、越靠前，贡献越大。"""
    score = 0
    for rank, doc_id in enumerate(retrieved_ids[:k], start=1):
        rel = gold.get(doc_id, 0)
        score += rel / math.log2(rank + 1)
    return score


def ndcg_at_k(retrieved_ids, gold, k=5):
    """nDCG@k = 当前排序的 DCG / 理想排序的 DCG。范围通常是 0 到 1。"""
    dcg = dcg_at_k(retrieved_ids, gold, k)
    ideal_rels = sorted(gold.values(), reverse=True)
    ideal_ids = [f"ideal_{i}" for i in range(len(ideal_rels))]
    ideal_gold = dict(zip(ideal_ids, ideal_rels))
    ideal_dcg = dcg_at_k(ideal_ids, ideal_gold, k)
    if ideal_dcg == 0:
        return 0
    return dcg / ideal_dcg


def evaluate_one(retrieved_ids, gold, k=5, relevant_threshold=2):
    """对单个 query 的检索结果计算四个指标。"""
    return {
        f"precision@{k}": precision_at_k(retrieved_ids, gold, k, relevant_threshold),
        f"recall@{k}": recall_at_k(retrieved_ids, gold, k, relevant_threshold),
        f"mrr@{k}": mrr_at_k(retrieved_ids, gold, k, relevant_threshold),
        f"ndcg@{k}": ndcg_at_k(retrieved_ids, gold, k),
    }